# TMDB 5000 Movie Dataset으로 장르기반 추천시스템 만들기

* 장르에 있는 텍스트를 벡터화 후 텍스트간 유사도를 구해 가까운 순서대로 정렬
* 유사도 척도: 코사인 유사도

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib

In [2]:
data = pd.read_csv("https://raw.githubusercontent.com/haram4th/ablearn/main/tmdb_5000_movies.csv")
data.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [7]:
data['genres'][0]

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [8]:
import json

In [11]:
temp = json.loads(data['genres'][0])
temp

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 878, 'name': 'Science Fiction'}]

In [20]:
result = []
for item in temp:
    result.append(item['name'])
result = " ".join(result)

In [21]:
result

'Action Adventure Fantasy Science Fiction'

In [18]:
data2 = data.copy()

In [25]:
data2.loc[0, 'genres'] = result

In [27]:
data2['production_countries']

0       [{"iso_3166_1": "US", "name": "United States o...
1       [{"iso_3166_1": "US", "name": "United States o...
2       [{"iso_3166_1": "GB", "name": "United Kingdom"...
3       [{"iso_3166_1": "US", "name": "United States o...
4       [{"iso_3166_1": "US", "name": "United States o...
                              ...                        
4798    [{"iso_3166_1": "MX", "name": "Mexico"}, {"iso...
4799                                                   []
4800    [{"iso_3166_1": "US", "name": "United States o...
4801    [{"iso_3166_1": "US", "name": "United States o...
4802    [{"iso_3166_1": "US", "name": "United States o...
Name: production_countries, Length: 4803, dtype: object

In [29]:
data.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [28]:
data.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

In [30]:
dict_cols = ['genres', 'keywords', 'production_companies', 'production_countries', 'spoken_languages']

In [37]:
data['genres'][:10]

0    [{"id": 28, "name": "Action"}, {"id": 12, "nam...
1    [{"id": 12, "name": "Adventure"}, {"id": 14, "...
2    [{"id": 28, "name": "Action"}, {"id": 12, "nam...
3    [{"id": 28, "name": "Action"}, {"id": 80, "nam...
4    [{"id": 28, "name": "Action"}, {"id": 12, "nam...
5    [{"id": 14, "name": "Fantasy"}, {"id": 28, "na...
6    [{"id": 16, "name": "Animation"}, {"id": 10751...
7    [{"id": 28, "name": "Action"}, {"id": 12, "nam...
8    [{"id": 12, "name": "Adventure"}, {"id": 14, "...
9    [{"id": 28, "name": "Action"}, {"id": 12, "nam...
Name: genres, dtype: object

In [46]:
for col in dict_cols:
    result = []
    for item in data[col]:
        temp_list = json.loads(item)
        temp_result = []
        for temp_item in temp_list:
            temp_result.append(temp_item['name'])
        result.append(" ".join(temp_result))
    data[col] = result

In [49]:
data['genres'][0]

'Action Adventure Fantasy Science Fiction'

In [50]:
data.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

In [51]:
data = data[['original_title', 'genres', 'popularity', 'runtime', 'vote_average', 'vote_count']]
data

,original_title,genres,popularity,runtime,vote_average,vote_count
0,Avatar,Action Adventure Fantasy Science Fiction,150.437577,162.0,7.2,11800
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action,139.082615,169.0,6.9,4500
2,Spectre,Action Adventure Crime,107.376788,148.0,6.3,4466
3,The Dark Knight Rises,Action Crime Drama Thriller,112.312950,165.0,7.6,9106
4,John Carter,Action Adventure Science Fiction,43.926995,132.0,6.1,2124
...,...,...,...,...,...,...
4798,El Mariachi,Action Crime Thriller,14.269792,81.0,6.6,238
4799,Newlyweds,Comedy Romance,0.642552,85.0,5.9,5
4800,"Signed, Sealed, Delivered",Comedy Drama Romance TV Movie,1.444476,120.0,7.0,6
4801,Shanghai Calling,,0.857008,98.0,5.7,7


In [52]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   original_title  4803 non-null   object 
 1   genres          4803 non-null   object 
 2   popularity      4803 non-null   float64
 3   runtime         4801 non-null   float64
 4   vote_average    4803 non-null   float64
 5   vote_count      4803 non-null   int64  
dtypes: float64(3), int64(1), object(2)
memory usage: 225.3+ KB


In [53]:
data = data.dropna()
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4801 entries, 0 to 4802
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   original_title  4801 non-null   object 
 1   genres          4801 non-null   object 
 2   popularity      4801 non-null   float64
 3   runtime         4801 non-null   float64
 4   vote_average    4801 non-null   float64
 5   vote_count      4801 non-null   int64  
dtypes: float64(3), int64(1), object(2)
memory usage: 262.6+ KB


# 장르 컬럼의 텍스트를 숫자로 벡터화

In [54]:
from sklearn.feature_extraction.text import CountVectorizer

In [55]:
cv = CountVectorizer(ngram_range=(1, 2))
genres_vec = cv.fit_transform(data['genres'])

In [56]:
print(len(cv.get_feature_names_out()))

276


In [58]:
genres_vec_df = pd.DataFrame(genres_vec)
genres_vec_df

,0
0,<Compressed Sparse Row sparse matrix of dtype ...
1,<Compressed Sparse Row sparse matrix of dtype ...
2,<Compressed Sparse Row sparse matrix of dtype ...
3,<Compressed Sparse Row sparse matrix of dtype ...
4,<Compressed Sparse Row sparse matrix of dtype ...
...,...
4796,<Compressed Sparse Row sparse matrix of dtype ...
4797,<Compressed Sparse Row sparse matrix of dtype ...
4798,<Compressed Sparse Row sparse matrix of dtype ...
4799,<Compressed Sparse Row sparse matrix of dtype ...


# 비슷한 장르를 찾기 위해서 코사인 유사도 구하기

In [59]:
from sklearn.metrics.pairwise import cosine_similarity

In [60]:
genres_sim = cosine_similarity(genres_vec, genres_vec)
print(genres_sim.shape)

(4801, 4801)


In [61]:
genres_sim_df = pd.DataFrame(genres_sim)
genres_sim_df 

,0,1,2,3,4,5,6,7,8,9,...,4791,4792,4793,4794,4795,4796,4797,4798,4799,4800
0,1.000000,0.596285,0.447214,0.125988,0.755929,0.596285,0.0,0.755929,0.447214,0.745356,...,0.000000,0.000000,0.000000,0.377964,0.000000,0.149071,0.0000,0.000000,0.0,0.0
1,0.596285,1.000000,0.400000,0.169031,0.338062,0.800000,0.0,0.338062,0.600000,0.800000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.200000,0.0000,0.000000,0.0,0.0
2,0.447214,0.400000,1.000000,0.338062,0.507093,0.600000,0.0,0.507093,0.200000,0.600000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.400000,0.0000,0.000000,0.0,0.0
3,0.125988,0.169031,0.338062,1.000000,0.142857,0.169031,0.0,0.142857,0.000000,0.169031,...,0.377964,0.169031,0.377964,0.428571,0.218218,0.676123,0.0000,0.125988,0.0,0.0
4,0.755929,0.338062,0.507093,0.142857,1.000000,0.507093,0.0,1.000000,0.169031,0.507093,...,0.000000,0.000000,0.000000,0.428571,0.000000,0.169031,0.0000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4796,0.149071,0.200000,0.400000,0.676123,0.169031,0.200000,0.0,0.169031,0.000000,0.200000,...,0.000000,0.200000,0.000000,0.169031,0.258199,1.000000,0.0000,0.000000,0.0,0.0
4797,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.258199,0.000000,0.000000,0.000000,0.000000,1.0000,0.384900,0.0,0.0
4798,0.000000,0.000000,0.000000,0.125988,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.333333,0.149071,0.333333,0.125988,0.000000,0.000000,0.3849,1.000000,0.0,0.0
4799,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.0,0.0


In [78]:
sorted_genres_sim = genres_sim.argsort()[:,::-1]

In [84]:
sorted_genres_sim 

array([[  14, 3493,   46, ..., 4763, 4764, 3571],
       [   1,   98, 2343, ...,   25, 4787, 4786],
       [   2, 1740, 1542, ..., 4757, 4758, 4799],
       ...,
       [4798, 3808, 3762, ...,   34,   33, 4799],
       [4800, 4799, 4798, ...,    2,    1,    0],
       [3353, 4696, 3625, ...,    2,    1,   15]], shape=(4801, 4801))

In [86]:
sorted_genres_sim[1,:]

array([   1,   98, 2343, ...,   25, 4787, 4786], shape=(4801,))

In [80]:
data.iloc[14,:]

original_title                                Man of Steel
genres            Action Adventure Fantasy Science Fiction
popularity                                       99.398009
runtime                                              143.0
vote_average                                           6.5
vote_count                                            6359
Name: 14, dtype: object

In [81]:
data.iloc[3493,:]

original_title    Beastmaster 2: Through the Portal of Time
genres             Action Adventure Fantasy Science Fiction
popularity                                         1.478505
runtime                                               107.0
vote_average                                            4.6
vote_count                                               17
Name: 3494, dtype: object

In [82]:
data.iloc[46,:]

original_title                  X-Men: Days of Future Past
genres            Action Adventure Fantasy Science Fiction
popularity                                      118.078691
runtime                                              131.0
vote_average                                           7.5
vote_count                                            6032
Name: 46, dtype: object

In [83]:
data.iloc[0,:]

original_title                                      Avatar
genres            Action Adventure Fantasy Science Fiction
popularity                                      150.437577
runtime                                              162.0
vote_average                                           7.2
vote_count                                           11800
Name: 0, dtype: object

# 영화 이름과 추천받을 개수를 입력받아 출력하는 함수

In [112]:
l1 = [14, 3493,   46,    0,  870,  813, 1296, 1652,  420,  419, 1191]
l1.remove(0)
l1

[14, 3493, 46, 870, 813, 1296, 1652, 420, 419, 1191]

In [118]:
searched_idx = data[data['original_title'] == 'Avatar'].index.values[0]
print("searched_idx", searched_idx)
recommanded_idx = sorted_genres_sim[searched_idx, :11]
recommanded_idx = recommanded_idx.tolist()
print(recommanded_idx)
if searched_idx in recommanded_idx:
    recommanded_idx.remove(searched_idx)
    print(recommanded_idx)
data.iloc[recommanded_idx, :]

searched_idx 0
[14, 3493, 46, 0, 870, 813, 1296, 1652, 420, 419, 1191]
[14, 3493, 46, 870, 813, 1296, 1652, 420, 419, 1191]


,original_title,genres,popularity,runtime,vote_average,vote_count
14,Man of Steel,Action Adventure Fantasy Science Fiction,99.398009,143.0,6.5,6359
3494,Beastmaster 2: Through the Portal of Time,Action Adventure Fantasy Science Fiction,1.478505,107.0,4.6,17
46,X-Men: Days of Future Past,Action Adventure Fantasy Science Fiction,118.078691,131.0,7.5,6032
870,Superman II,Action Adventure Fantasy Science Fiction,30.515175,127.0,6.5,629
813,Superman,Action Adventure Fantasy Science Fiction,48.507081,143.0,6.9,1022
1296,Superman III,Comedy Action Adventure Fantasy Science Fiction,22.164202,125.0,5.3,490
1652,Dragonball Evolution,Action Adventure Fantasy Science Fiction Thriller,21.677732,85.0,2.9,462
420,Hellboy II: The Golden Army,Adventure Fantasy Science Fiction,58.579760,120.0,6.5,1527
419,Jumper,Adventure Fantasy Science Fiction,21.218000,88.0,5.9,1799
1191,Small Soldiers,Comedy Adventure Fantasy Science Fiction Action,23.088571,110.0,6.2,511


In [103]:
recommanded_idx

array([  14, 3493,   46,    0,  870,  813, 1296, 1652,  420,  419, 1191])

In [135]:
def find_sim_movie(data, sorted_genres_sim, title_name, top_n=11):
    searched_idx = data[data['original_title'] == title_name].index.values[0]
    print(searched_idx)
    recommanded_idx = sorted_genres_sim[searched_idx, :top_n+1]
    recommanded_idx = recommanded_idx.tolist()
    print(recommanded_idx)
    if searched_idx in recommanded_idx:
        recommanded_idx.remove(searched_idx)
        print(recommanded_idx)
    return data.loc[recommanded_idx, :]

In [136]:
data

,original_title,genres,popularity,runtime,vote_average,vote_count
0,Avatar,Action Adventure Fantasy Science Fiction,150.437577,162.0,7.2,11800
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action,139.082615,169.0,6.9,4500
2,Spectre,Action Adventure Crime,107.376788,148.0,6.3,4466
3,The Dark Knight Rises,Action Crime Drama Thriller,112.312950,165.0,7.6,9106
4,John Carter,Action Adventure Science Fiction,43.926995,132.0,6.1,2124
...,...,...,...,...,...,...
4798,El Mariachi,Action Crime Thriller,14.269792,81.0,6.6,238
4799,Newlyweds,Comedy Romance,0.642552,85.0,5.9,5
4800,"Signed, Sealed, Delivered",Comedy Drama Romance TV Movie,1.444476,120.0,7.0,6
4801,Shanghai Calling,,0.857008,98.0,5.7,7


In [137]:
find_sim_movie(data, sorted_genres_sim, "Spectre", top_n=5)

2
[2, 1740, 1542, 1082, 969, 873]
[1740, 1542, 1082, 969, 873]


,original_title,genres,popularity,runtime,vote_average,vote_count
1740,Kick-Ass 2,Action Adventure Crime,40.286350,103.0,6.3,2224
1542,Speed,Action Adventure Crime,49.526736,116.0,6.8,1783
1082,16 Blocks,Action Adventure Crime Thriller,32.310220,105.0,6.2,661
969,Assassins,Action Adventure Crime Thriller,23.073933,132.0,6.0,387
873,Shaft,Action Adventure Crime Thriller,19.643365,99.0,5.5,308


# 영화이름 입력받아서 비슷한 장르 영화 추천받기

In [139]:
movie_name = input("영화 이름을 입력하세요: ")
find_sim_movie(data, sorted_genres_sim, movie_name, top_n=5)

영화 이름을 입력하세요: Avatar
0
[14, 3493, 46, 0, 870, 813]
[14, 3493, 46, 870, 813]


,original_title,genres,popularity,runtime,vote_average,vote_count
14,Man of Steel,Action Adventure Fantasy Science Fiction,99.398009,143.0,6.5,6359
3493,Morvern Callar,Drama,2.507912,97.0,7.2,34
46,X-Men: Days of Future Past,Action Adventure Fantasy Science Fiction,118.078691,131.0,7.5,6032
870,Superman II,Action Adventure Fantasy Science Fiction,30.515175,127.0,6.5,629
813,Superman,Action Adventure Fantasy Science Fiction,48.507081,143.0,6.9,1022
